# LLM-based Natural Language Interface to Databases

This notebook implements a prototype Natural Language Interface to Databases using large language models.

The pipeline performs Text-to-SQL generation, SQL-to-Text explanation, SQL execution over SQLite databases from the Spider dataset, and Query Results-to-Text verbalization.

The goal is to demonstrate how relational databases can be queried and explained through natural language using LLM-based components.

In [ ]:
import os

LLM_BACKEND = "openai"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if LLM_BACKEND == "openai" and not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY is not set. Please set it as an environment variable before running the notebook."
    )
# LLM_BACKEND = "vllm"

## Install Dependencies

In [2]:
if LLM_BACKEND == "vllm":
  !pip install vllm bitsandbytes>=0.44.0 tabulate
elif LLM_BACKEND == "openai":
  !pip install openai
else:
  raise ValueError(f"LLM backend '{LLM_BACKEND}' not supported!")

## Prepare LLM

In [3]:
import sys

!{sys.executable} -m pip install --upgrade openai

In [4]:
if LLM_BACKEND == "vllm":
    from vllm import LLM, SamplingParams
    import torch

    # If using the official Llama checkpoint, access/login may be required:
    # model_id = "meta-llama/Llama-3.1-8B-Instruct"

    model_id = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

    llm = LLM(
        model=model_id,
        dtype=torch.float16,
        trust_remote_code=True,
        quantization="bitsandbytes"
    )

    def vllm_generate(chat):
        sampling_params = SamplingParams(max_tokens=512)
        outputs = llm.chat(chat, sampling_params=sampling_params)
        generated_text = outputs[0].outputs[0].text
        return generated_text

    llm_generate = vllm_generate


elif LLM_BACKEND == "openai":
    from openai import OpenAI
    from typing import Dict, List

    def gpt_generate(chat: List[Dict[str, str]]):
        client = OpenAI(api_key=OPENAI_API_KEY)

        completion = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=chat
        )

        generated_text = completion.choices[0].message.content
        return generated_text

    llm_generate = gpt_generate


else:
    raise ValueError(f"LLM backend '{LLM_BACKEND}' not supported!")

## Aditional imports and helper functions

In [5]:
from IPython.display import display, Markdown

## Download the Spider dataset

In [6]:
# Download and extract the Spider dataset automatically

import zipfile
from pathlib import Path
import json
import pandas as pd

%pip install -q gdown

import gdown

DATA_DIR = Path("spider_data")
ZIP_PATH = Path("spider_data.zip")

# Correct official Spider dataset Google Drive file ID
file_id = "1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J"
url = f"https://drive.google.com/uc?id={file_id}"

# Download zip if needed
if not ZIP_PATH.exists():
    gdown.download(url, str(ZIP_PATH), quiet=False)

# Extract zip if needed
if not DATA_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(".")

# Load dev set
dev_path = DATA_DIR / "dev.json"
tables_path = DATA_DIR / "tables.json"

spider_dev = pd.read_json(dev_path)
spider_dev = spider_dev[["question", "query", "db_id"]]

# Load DB schemas
with open(tables_path, "r", encoding="utf-8") as fp:
    schema_list = json.load(fp)

schema_dict = {schema["db_id"]: schema for schema in schema_list}

spider_dev.head()

Note: you may need to restart the kernel to use updated packages.


,question,query,db_id
0,How many singers do we have?,SELECT count(*) FROM singer,concert_singer
1,What is the total number of singers?,SELECT count(*) FROM singer,concert_singer
2,"Show name, country, age for all singers ordere...","SELECT name , country , age FROM singer ORDE...",concert_singer
3,"What are the names, countries, and ages for ev...","SELECT name , country , age FROM singer ORDE...",concert_singer
4,"What is the average, minimum, and maximum age ...","SELECT avg(age) , min(age) , max(age) FROM s...",concert_singer


## Try usign the LLM

Before delving into NLIDBs, let us first take a look on how we use an LLM.
When we prompt the LLM, we give it a the start of a conversation and ask it to generate a response.
In the example below we have two parts in the conversation:

- A **"system"** message that we call `instructions`: System messages are introductory information that describe the role of the LLM and how it should behave. In this example we specify that the LLM is a pirate and it should respond in such a manner.

- A **"user"** message that we call `question`: User messages are parts of the conversation that were expressed by the user (i.e., you). These messages are usually asking the LLM to perform a task or present a question that should be answered.

There is an additional message role called **"assistant"**: This role indicates that this part of the conversation is a response that was generated by the assistant (i.e., the LLM).
This is useful when we have a long conversation with multiple question-answer turns, or when we want to perform few-shot prompting.
We will se an example usage of this role later in the exercise with few-shot prompting.



In [7]:
instructions =\
"""
You are a concise technical assistant. Explain what Text-to-SQL is in two sentences.
"""

question =\
"""
Can you tell me a few things about Text-to-SQL?
"""

conversation = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]

generated_text = llm_generate(conversation)

display(generated_text)

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

# Part 1: Text-to-SQL

## 1.1: Getting familiar with the data

The first task we will explore is Text-to-SQL, which can enable NL querying over a database.
We have already loaded the Spider dataset - a famous Text-to-SQL dataset - for you to try.

In the code cells bellow you can get familiar with the dataset and its structure.
Take a look at the two loaded variables:
- `spider_dev`: A dataframe that contains all the NL/SQL pairs of the dev split of the dataset and the id of the database that it corresponds to.
- `schema_dict`: A dictionary mapping database ids to the corresponding database info (table names, column names, etc.)

In [8]:
# 1. Look at the first rows of the Spider dev dataset
display(spider_dev.head())

# 2. Pick one example
example = spider_dev.iloc[0]

# 3. Print the natural language question
print("Question:")
print(example["question"])

# 4. Print the corresponding SQL query
print("\nSQL query:")
print(example["query"])

# 5. Print the database id
db_id = example["db_id"]
print("\nDatabase ID:")
print(db_id)

# 6. Load the corresponding schema
schema = schema_dict[db_id]

# 7. Inspect the schema keys
print("\nSchema keys:")
print(schema.keys())

# 8. Print table names
print("\nTables:")
print(schema["table_names_original"])

# 9. Print column names
print("\nColumns:")
print(schema["column_names_original"])

,question,query,db_id
0,How many singers do we have?,SELECT count(*) FROM singer,concert_singer
1,What is the total number of singers?,SELECT count(*) FROM singer,concert_singer
2,"Show name, country, age for all singers ordere...","SELECT name , country , age FROM singer ORDE...",concert_singer
3,"What are the names, countries, and ages for ev...","SELECT name , country , age FROM singer ORDE...",concert_singer
4,"What is the average, minimum, and maximum age ...","SELECT avg(age) , min(age) , max(age) FROM s...",concert_singer


Question:
How many singers do we have?

SQL query:
SELECT count(*) FROM singer

Database ID:
concert_singer

Schema keys:
dict_keys(['column_names', 'column_names_original', 'column_types', 'db_id', 'foreign_keys', 'primary_keys', 'table_names', 'table_names_original'])

Tables:
['stadium', 'singer', 'concert', 'singer_in_concert']

Columns:
[[-1, '*'], [0, 'Stadium_ID'], [0, 'Location'], [0, 'Name'], [0, 'Capacity'], [0, 'Highest'], [0, 'Lowest'], [0, 'Average'], [1, 'Singer_ID'], [1, 'Name'], [1, 'Country'], [1, 'Song_Name'], [1, 'Song_release_year'], [1, 'Age'], [1, 'Is_male'], [2, 'concert_ID'], [2, 'concert_Name'], [2, 'Theme'], [2, 'Stadium_ID'], [2, 'Year'], [3, 'concert_ID'], [3, 'Singer_ID']]


In [9]:
spider_dev

,question,query,db_id
0,How many singers do we have?,SELECT count(*) FROM singer,concert_singer
1,What is the total number of singers?,SELECT count(*) FROM singer,concert_singer
2,"Show name, country, age for all singers ordere...","SELECT name , country , age FROM singer ORDE...",concert_singer
3,"What are the names, countries, and ages for ev...","SELECT name , country , age FROM singer ORDE...",concert_singer
4,"What is the average, minimum, and maximum age ...","SELECT avg(age) , min(age) , max(age) FROM s...",concert_singer
...,...,...,...
1029,What are the citizenships that are shared by s...,SELECT Citizenship FROM singer WHERE Birth_Yea...,singer
1030,How many available features are there in total?,SELECT count(*) FROM Other_Available_Features,real_estate_properties
1031,What is the feature type name of feature AirCon?,SELECT T2.feature_type_name FROM Other_Availab...,real_estate_properties
1032,Show the property type descriptions of propert...,SELECT T2.property_type_description FROM Prope...,real_estate_properties


In [10]:
schema_dict["flight_2"]

{'column_names': [[-1, '*'],
  [0, 'airline id'],
  [0, 'airline name'],
  [0, 'abbreviation'],
  [0, 'country'],
  [1, 'city'],
  [1, 'airport code'],
  [1, 'airport name'],
  [1, 'country'],
  [1, 'country abbrev'],
  [2, 'airline'],
  [2, 'flight number'],
  [2, 'source airport'],
  [2, 'destination airport']],
 'column_names_original': [[-1, '*'],
  [0, 'uid'],
  [0, 'Airline'],
  [0, 'Abbreviation'],
  [0, 'Country'],
  [1, 'City'],
  [1, 'AirportCode'],
  [1, 'AirportName'],
  [1, 'Country'],
  [1, 'CountryAbbrev'],
  [2, 'Airline'],
  [2, 'FlightNo'],
  [2, 'SourceAirport'],
  [2, 'DestAirport']],
 'column_types': ['text',
  'number',
  'text',
  'text',
  'text',
  'text',
  'text',
  'text',
  'text',
  'text',
  'number',
  'number',
  'text',
  'text'],
 'db_id': 'flight_2',
 'foreign_keys': [[13, 6], [12, 6]],
 'primary_keys': [1, 6, 10],
 'table_names': ['airlines', 'airports', 'flights'],
 'table_names_original': ['airlines', 'airports', 'flights']}

## 1.2: Creating a first LLM prompt for Text-to-SQL
Now that you are more familiar with the data, let's try to use a LLM for Text-to-SQL.
In the code block bellow, you must complete the intructions and question parts of the prompt with all the necessary info for the LLM to generate the SQL.
Similarly to the introductory example, the `instructions` are more general guidelines about how the LLM should behave (e.g., "You are an expert in SQL", "You only respond with correct SQL queries", etc.), while the `question` contains all the information that is relevant to the specific example of the task it must solve.

In [ ]:
instructions = """
You are an expert Text-to-SQL assistant.
Your task is to translate natural language questions into valid SQL queries.

Rules:
- Use only the tables and columns provided in the database schema.
- Do not invent table names or column names.
- Return only the SQL query.
- Do not include explanations, comments, markdown, or any extra text.
"""

question = """
Database schema:

Table: stadium
Columns: Stadium_ID, Location, Name, Capacity, Highest, Lowest, Average

Table: singer
Columns: Singer_ID, Name, Country, Song_Name, Song_release_year, Age, Is_male

Table: concert
Columns: concert_ID, concert_Name, Theme, Stadium_ID, Year

Table: singer_in_concert
Columns: concert_ID, Singer_ID

Natural language question:
How many singers do we have?

Generate the corresponding SQL query.
"""

conversation = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = llm_generate(conversation)
print(response)

print("\nGold SQL:")
print(example["query"])

## 1.3: Adding a few-shot example to the prompt
In this section we will see how we can add a demonstration example to the LLM's prompt.
A demonstartion exaple consists of a question similar to the one you created in the previous task and the reponse that an ideal model would generate.
Such demonstration examples can help the model in two ways:

1. Provide a clear example of the format in which the model is expected to respond. For instance, if we give an example where the response contains an SQL query and a text exaplanation of how it was created, then the model will most likely also include an explanation in its answer. On the contrary, if the example response only contains the SQL, the model will most likely only generate a SQL query.
2. Provide examples that are similar to the specific question we want the model to answer, to help it generate a more accurate response. Multiple studies have shown that if we dynamically retrieve more similar examples to each given question, the LLM's performance improves.

In [ ]:
instructions = """
You are an expert Text-to-SQL assistant.
Your task is to translate natural language questions into valid SQL queries.

Rules:
- Use only the tables and columns provided in the database schema.
- Do not invent table names or column names.
- Return only the SQL query.
- Do not include explanations, comments, markdown, or any extra text.
"""

example_question = """
Database schema:

Table: stadium
Columns: Stadium_ID, Location, Name, Capacity, Highest, Lowest, Average

Table: singer
Columns: Singer_ID, Name, Country, Song_Name, Song_release_year, Age, Is_male

Table: concert
Columns: concert_ID, concert_Name, Theme, Stadium_ID, Year

Table: singer_in_concert
Columns: concert_ID, Singer_ID

Natural language question:
What are the names of all singers?

Generate the corresponding SQL query.
"""

example_response = """
SELECT Name FROM singer;
"""

question = """
Database schema:

Table: stadium
Columns: Stadium_ID, Location, Name, Capacity, Highest, Lowest, Average

Table: singer
Columns: Singer_ID, Name, Country, Song_Name, Song_release_year, Age, Is_male

Table: concert
Columns: concert_ID, concert_Name, Theme, Stadium_ID, Year

Table: singer_in_concert
Columns: concert_ID, Singer_ID

Natural language question:
How many singers do we have?

Generate the corresponding SQL query.
"""

conversation = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": example_question},
    {"role": "assistant", "content": example_response},
    {"role": "user", "content": question},
]

generated_text = llm_generate(conversation)
print(generated_text)

# Part 2: SQL-to-Text

## 2.1: Creating a LLM prompt for SQL-to-Text
Now that you are familiar with Text-to-SQL, it's time to move on to the reverse task of SQL-to-Text.
The goal here is to explain a SQL query in natural language so that the user can understand what it does and be confident that it matches their needs.

In the code block bellow, you must complete the intructions and question parts of the prompt with all the necessary info for the LLM to generate the NL explanation of a SQL.
Similarly to the previous part, you must complete the `instructions` and `question` strings so that they contain all the necessary information for the LLM to generate the desired explanation.

## 2.2: Changing the expression of the explanation with Prompt Engineering

Compared to the strictness of SQL, natural language allows us to express the same thing in many different ways.
This means that a query explanation can be phrased in many different ways.
However depending on the user and the use case some expressions might be preferable.
For example, in the context of NLIDBs, we want to provide a brief and compact explanation rather than a long walkthrough of every part of the query.
This is similar to the way NL questions are expressed in the Spider dataset.

One way to achieve this when using a LLM is to provide specific instruction on how we want the LLM to generate the explanation.

## 2.2: Changing the expression of the explanation with Few-Shot Demonstrations

As we saw earlier, another way to guide the LLM's behavior towards our goal is to provide similar examples.

# Part 3: Data-to-Text

**Data-to-Text** is a general term that describes taking structured data as input and producing natural language that describes them. The two most explored structures in Data-to-Text are tables (Table-to-Text) and graphs (Graph-to-Text).

However, since this notebook focuses on NLIDBs we will explore **Query Results-to-Text** that describes in text the results of an executed query.

In [11]:
import pandas as pd
import sqlite3

def dataframe_to_markdown(df: pd.DataFrame) -> str:
    """ Utility method to serialize a dataframe to a markdown table """
    return df.to_markdown(index=False)


def execute_query(db_path: str, query: str) -> pd.DataFrame:
    """ Utility method to execute a query on a sqlite database """
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query(query, conn)

    return df


## 3.1 Creating the Data-to-Text Prompt

Similar to the previous components we have to define an instruction prompt (system) and the template prompt that we are going to fill with the query results.

In [12]:
dataframe = pd.DataFrame(data={"Country": ["USA", "USSR", "China", "Great Britain", "France"], "Gold": [1105, 395, 303, 298, 239], "Silver": [879, 319, 226, 340, 277], "Bronze": [781, 296, 198, 343, 299]})

display(dataframe)

,Country,Gold,Silver,Bronze
0,USA,1105,879,781
1,USSR,395,319,296
2,China,303,226,198
3,Great Britain,298,340,343
4,France,239,277,299


In [ ]:
import os
from getpass import getpass

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    OPENAI_API_KEY = input("Paste your OpenAI API key: ").strip()

In [ ]:
dataframe = pd.DataFrame(
    data={
        "Country": ["USA", "USSR", "China", "Great Britain", "France"],
        "Gold": [1105, 395, 303, 298, 239],
        "Silver": [879, 319, 226, 340, 277],
        "Bronze": [781, 296, 198, 343, 299],
    }
)

instructions = """
You are an expert Data-to-Text assistant.
Your goal is to convert structured tabular data into a clear, accurate natural language description.

Rules:
- Describe only the information that is present in the table.
- Do not invent facts or add external knowledge.
- Mention the main trends and comparisons in the data.
- Keep the answer concise but informative.
- Use natural language, not SQL or code.
"""

question = f"""
The following table contains information about Olympic medals for the countries with the most gold medals.

Table:
{dataframe_to_markdown(dataframe)}

Write a clear natural language description of the table.
Mention which country has the most gold medals, compare the countries briefly, and summarize the main pattern in the medal counts.
"""

conversation = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]

generated_text = llm_generate(conversation)

print(generated_text)

## 3.2 Query Results from an executed query

In a scenario closer to real world, we would not have hardcoded examples but instead we would execute a query (written by a user or generated from a Text-to-SQL model).

In [ ]:
from pathlib import Path

DATA_DIR = Path("spider_data")
db_path = DATA_DIR / "database" / "cinema" / "cinema.sqlite"
db_path = str(db_path)

query = """
SELECT
    Directed_by,
    COUNT(*) AS number_of_films
FROM film
GROUP BY Directed_by
ORDER BY number_of_films DESC
LIMIT 5;
"""

result_df = execute_query(db_path, query)

print("Executed SQL query:")
print(query)

print("\nQuery results:")
display(result_df)

results_markdown = dataframe_to_markdown(result_df)

print("\nQuery results as markdown:")
print(results_markdown)

instructions = """
You are an expert Query Results-to-Text assistant.
Your goal is to convert the results of an executed SQL query into a clear and accurate natural language description.

Rules:
- Describe only the information present in the query results.
- Do not invent facts or add external knowledge.
- Use the SQL query and table names only as context.
- Mention the main comparison shown in the result table.
- Keep the answer concise and informative.
- Use natural language, not SQL or code.
"""

question = f"""
The following SQL query was executed on the cinema database.

SQL query:
{query}

Tables used:
film

Query results:
{results_markdown}

Write a clear natural language description of the query results.
"""

conversation = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]

generated_text = llm_generate(conversation)

print("\nGenerated verbalisation:")
print(generated_text)

print("\nObservation:")
print(
    "Providing the SQL query and the table name improves the verbalisation because "
    "the model can understand that the result table shows the directors with the highest "
    "number of films, instead of merely describing isolated rows."
)

# Part 4: Combining everything together

In [ ]:
import os
import time
import re

def get_db_id_from_path(db_path):
    """
    Extract the database id from a Spider database path.
    Example:
    /content/spider_data/database/cinema/cinema.sqlite -> cinema
    """
    return os.path.basename(os.path.dirname(db_path))


def format_schema_for_prompt(db_id):
    """
    Convert the Spider schema into a readable text format for prompting.
    """
    schema = schema_dict[db_id]
    table_names = schema["table_names_original"]
    column_names = schema["column_names_original"]

    table_to_columns = {table: [] for table in table_names}

    for table_idx, column_name in column_names:
        if table_idx == -1:
            continue
        table_name = table_names[table_idx]
        table_to_columns[table_name].append(column_name)

    schema_text = ""
    for table_name, columns in table_to_columns.items():
        schema_text += f"Table: {table_name}\n"
        schema_text += f"Columns: {', '.join(columns)}\n\n"

    return schema_text.strip()


def clean_generated_sql(generated_sql):
    """
    Remove markdown formatting or extra text if the model returns it.
    """
    sql = generated_sql.strip()

    sql = re.sub(r"```sql", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"```", "", sql)
    sql = sql.strip()

    return sql


def text_to_sql(natural_language_question, db_path):
    """
    Component 1: Text-to-SQL.
    Translates a natural language question into SQL using the database schema.
    """
    db_id = get_db_id_from_path(db_path)
    schema_text = format_schema_for_prompt(db_id)

    instructions = """
You are an expert Text-to-SQL assistant.
Your task is to translate natural language questions into valid SQLite SQL queries.

Rules:
- Use only the tables and columns provided in the database schema.
- Do not invent table names or column names.
- Return only the SQL query.
- Do not include explanations, comments, markdown, or extra text.
"""

    question = f"""
Database schema:

{schema_text}

Natural language question:
{natural_language_question}

Generate the corresponding SQLite SQL query.
"""

    conversation = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ]

    generated_sql = llm_generate(conversation)
    generated_sql = clean_generated_sql(generated_sql)

    return generated_sql


def sql_to_text(sql_query):
    """
    Component 2: SQL-to-Text.
    Explains the generated SQL query in natural language.
    """
    instructions = """
You are an expert SQL-to-Text assistant.
Your task is to explain SQL queries in clear natural language.

Rules:
- Explain what the query retrieves or computes.
- Be concise and accurate.
- Do not execute the query.
- Do not invent information that is not implied by the SQL query.
"""

    question = f"""
SQL query:
{sql_query}

Explain what this SQL query does in natural language.
"""

    conversation = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ]

    explanation = llm_generate(conversation)

    return explanation


def query_results_to_text(sql_query, result_df, db_path):
    """
    Component 3: Query Results-to-Text.
    Converts executed query results into natural language.
    """
    db_id = get_db_id_from_path(db_path)
    results_markdown = dataframe_to_markdown(result_df)

    instructions = """
You are an expert Query Results-to-Text assistant.
Your task is to convert SQL query results into a clear natural language answer.

Rules:
- Describe only the information present in the query results.
- Do not invent facts or add external knowledge.
- Use the SQL query only as context.
- If the result is empty, say that the query returned no rows.
- Keep the answer concise and informative.
- Use natural language, not SQL or code.
"""

    question = f"""
Database id:
{db_id}

SQL query:
{sql_query}

Query results:
{results_markdown}

Write a clear natural language answer based on the query results.
"""

    conversation = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ]

    answer = llm_generate(conversation)

    return answer


def nlidb_pipeline(natural_language_question, db_path):
    """
    Full NLIDB pipeline:
    1. Text-to-SQL
    2. SQL-to-Text
    3. SQL execution
    4. Query Results-to-Text
    """

    timings = {}

    total_start = time.time()

    # 1. Text-to-SQL
    start = time.time()
    generated_sql = text_to_sql(natural_language_question, db_path)
    timings["text_to_sql_seconds"] = time.time() - start

    # 2. SQL-to-Text
    start = time.time()
    sql_explanation = sql_to_text(generated_sql)
    timings["sql_to_text_seconds"] = time.time() - start

    # 3. Execute SQL query
    start = time.time()
    result_df = execute_query(db_path, generated_sql)
    timings["query_execution_seconds"] = time.time() - start

    # 4. Query Results-to-Text
    start = time.time()
    result_text = query_results_to_text(generated_sql, result_df, db_path)
    timings["results_to_text_seconds"] = time.time() - start

    timings["total_pipeline_seconds"] = time.time() - total_start

    return {
        "natural_language_question": natural_language_question,
        "db_path": db_path,
        "generated_sql": generated_sql,
        "sql_explanation": sql_explanation,
        "query_results": result_df,
        "result_text": result_text,
        "timings": timings,
    }


# Example run on the cinema database
from pathlib import Path

DATA_DIR = Path("spider_data")
db_path = DATA_DIR / "database" / "cinema" / "cinema.sqlite"
db_path = str(db_path)

natural_language_question = "How many films did each director direct?"

output = nlidb_pipeline(natural_language_question, db_path)

print("Natural language question:")
print(output["natural_language_question"])

print("\nGenerated SQL:")
print(output["generated_sql"])

print("\nSQL-to-Text explanation:")
print(output["sql_explanation"])

print("\nQuery results:")
display(output["query_results"])

print("\nQuery Results-to-Text answer:")
print(output["result_text"])

print("\nLatency measurements:")
for component, seconds in output["timings"].items():
    print(f"{component}: {seconds:.3f} seconds")

## Findings

The pipeline combines the three main components: Text-to-SQL, SQL-to-Text, and Query Results-to-Text. The user gives a natural language question, the system generates SQL, explains the SQL, executes it, and then describes the results in natural language.

### Text-to-SQL performance

The Text-to-SQL step worked correctly for the tested example. The model generated a valid SQL query using the correct table and column names. This step depends a lot on how clearly the database schema is given to the model. If the schema is unclear, the model may choose wrong tables or columns.

### SQL-to-Text performance

The SQL-to-Text step also worked well. It correctly explained that the query counts how many films each director directed. This step is easier because the SQL query is already structured.

### Query Results-to-Text performance

The Query Results-to-Text step produced a clear answer from the query results. It correctly stated that Bill Schreiner directed 4 films and Jesus Salvador Treviño directed 1 film.

### Combined pipeline performance

Overall, the full pipeline worked correctly for this example. However, the most important weakness is error propagation. If the Text-to-SQL step generates a wrong SQL query, then the explanation, execution results, and final verbalisation will also be wrong.

### Latency

The total latency was about 6.9 seconds. Most of the time was spent on the LLM calls, especially the Text-to-SQL step. The actual database query execution was very fast.

### Possible improvements

The system could be improved by giving better schema information, adding few-shot examples, validating the generated SQL before execution, and using execution errors to repair wrong queries. Latency could also be improved by reducing the number of LLM calls or combining some steps together.